# Test Project Structure and Configuration

## Organizing Your Tests
---

In this lesson, you'll learn how to **properly organize test projects** and configure pytest for your needs.

You have tests, but where do they go? How do you share common setup? Let's find out.

1. [Directory Structure](#Directory-Structure),
    - [Standard Layout](#Standard-Layout),
    - [Alternative Layouts](#Alternative-Layouts),
    - [Naming Conventions](#Naming-Conventions),
2. [The conftest.py File](#The-conftest.py-File),
    - [What Is conftest.py](#What-Is-conftest.py),
    - [Shared Fixtures](#Shared-Fixtures),
    - [Fixture Scope](#Fixture-Scope),
    - [Multiple conftest Files](#Multiple-conftest-Files),
3. [Test Functions vs Test Classes](#Test-Functions-vs-Test-Classes),
    - [When to Use Functions](#When-to-Use-Functions),
    - [When to Use Classes](#When-to-Use-Classes),
    - [Mixing Both](#Mixing-Both),
4. [pytest Configuration](#pytest-Configuration),
    - [pyproject.toml](#pyproject.toml),
    - [pytest.ini](#pytest.ini),
    - [Common Options](#Common-Options),
5. [Running Specific Tests](#Running-Specific-Tests),
    - [By File and Directory](#By-File-and-Directory),
    - [By Name Pattern](#By-Name-Pattern),
    - [By Marker](#By-Marker),
6. [Practical Example: E-commerce Test Suite](#Practical-Example:-E-commerce-Test-Suite),
7. [🧠 EXERCISE 🧠, Organize a Test Suite](#🧠-EXERCISE-🧠,-Organize-a-Test-Suite).

<br>

## Directory Structure

---

A well-organized test structure makes your project maintainable and your tests easy to find.

<br>

### Standard Layout

---

The most common and recommended structure for Python project (without Django):

```
my_project/
├── src/
│   └── script.py             # Source code
├── tests/
│   ├── conftest.py           # Shared fixtures
│   └── test_script.py        # Tests for script.py
├── pyproject.toml
└── .venv/
```

This is also the setup we use in this course:

```
kbc-django-copy/
├── src/
│   └── script.py
├── tests/
│   ├── conftest.py
│   └── test_script.py
└── pyproject.toml
```

**Key points:**

- Tests are in a separate `tests/` directory at project root
- Test files mirror the source structure
- `conftest.py` contains shared test utilities
- Configuration in `pyproject.toml`

<br>

### Alternative Layouts

---

**Nested tests (for larger projects):**

```
my_project/
├── src/
│   └── my_package/
│       ├── auth/
│       │   ├── __init__.py
│       │   ├── login.py
│       │   └── permissions.py
│       └── orders/
│           ├── __init__.py
│           ├── models.py
│           └── processing.py
├── tests/
│   ├── conftest.py              # Root-level fixtures
│   ├── auth/
│   │   ├── conftest.py          # Auth-specific fixtures
│   │   ├── test_login.py
│   │   └── test_permissions.py
│   └── orders/
│       ├── conftest.py          # Order-specific fixtures
│       ├── test_models.py
│       └── test_processing.py
└── pyproject.toml
```

**Tests alongside source (less common):**

```
my_project/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── models.py
│       ├── test_models.py    # Tests next to source
│       ├── services.py
│       └── test_services.py
└── pyproject.toml
```

This approach is less common but can be useful for small projects or when tests are tightly coupled to implementation.

<br>

### Naming Conventions

---

pytest auto-discovers tests based on these conventions:

| Element | Convention | Example |
|---------|------------|----------|
| **Test directories** | `tests/`, `test/` | `tests/` |
| **Test files** | `test_*.py`, `*_test.py` | `test_users.py` |
| **Test functions** | `test_*` | `test_create_user()` |
| **Test classes** | `Test*` (no `__init__`) | `TestUserCreation` |
| **Test methods** | `test_*` | `test_valid_email()` |

In [ ]:
# Good naming examples

# test_users.py - tests for users module
def test_create_user_with_valid_data():
    """Test user creation with valid input."""
    pass

def test_create_user_rejects_duplicate_email():
    """Test that duplicate emails are rejected."""
    pass

def test_user_password_is_hashed():
    """Test that passwords are properly hashed."""
    pass


class TestUserPermissions:
    """Tests for user permission system."""
    
    def test_admin_has_all_permissions(self):
        pass
    
    def test_regular_user_has_limited_permissions(self):
        pass

<br>

## The `conftest.py` File

---

### What Is conftest.py

---

`conftest.py` is a special file implicitly recognized by pytest. It contains:

- **Fixtures** - reusable test setup/teardown
- **Hooks** - customize pytest behavior
- **Plugins** - local pytest plugins

**Key features:**
- No need to import - fixtures are auto-discovered
- Can have multiple `conftest.py` files at different levels
- Fixtures are available to all tests in same directory and below

<br>

### Shared Fixtures

---

A **fixture** provides test data or test setup that can be reused across tests:

In [ ]:
# conftest.py  ← in the same directory as the test_script.py

import pytest


@pytest.fixture
def sample_blog_post() -> dict:
    """Provide a sample blog post for testing."""
    return {
        "title": "My First Blog Post",
        "author": "John Doe",
        "content": "This is a sample content for testing purposes only.",
    }


@pytest.fixture
def sample_comment() -> dict:
    """Provide a sample comment for testing."""
    return {
        "content": "Great post, very informative and well written!",
    }

In [ ]:
# tests/test_script.py

def test_blog_post_has_title(sample_blog_post):
    """Test using the sample_blog_post fixture."""
    assert sample_blog_post["title"] == "My First Blog Post"
    assert sample_blog_post["author"] == "John Doe"


def test_blog_post_content_not_empty(sample_blog_post):
    """Another test using the same fixture."""
    assert len(sample_blog_post["content"]) > 0

<br>

**Fixtures with setup and teardown:**

You will use setup and teardown methods when your test requires initial preparation—like creating a temporary database user or opening a file. Once the test finishes, teardown handles the cleanup so that leftover data doesn't break other tests. (Note: In modern pytest, this preparation and cleanup is usually handled more elegantly using fixtures

In [ ]:
# tests/conftest.py

import pytest
import tempfile
import os


@pytest.fixture
def temp_config_file():
    """Create a temporary config file, clean up after test."""
    # Setup: create temp file
    fd, path = tempfile.mkstemp(suffix=".json")
    with os.fdopen(fd, "w") as f:
        f.write('{"debug": true, "timeout": 30}')
    
    # Provide the path to the test
    yield path
    
    # Teardown: remove temp file
    if os.path.exists(path):
        os.remove(path)

In [ ]:
# tests/test_config.py

import json


def test_load_config(temp_config_file):
    """Test config loading from file."""
    with open(temp_config_file) as f:
        config = json.load(f)
    
    assert config["debug"] == True
    assert config["timeout"] == 30
    
    # After test: file is automatically cleaned up

<br>

### Fixture Scope

---

Fixtures can have different scopes controlling when they're created/destroyed:

| Scope | When Created | When Destroyed | Use Case |
|-------|--------------|----------------|----------|
| `function` (default) | Before each test | After each test | Test isolation |
| `class` | Before first test in class | After last test in class | Class-level setup |
| `module` | Before first test in file | After last test in file | Expensive setup |
| `session` | Once at start | Once at end | Database connection |

In [ ]:
# tests/conftest.py

import pytest


@pytest.fixture(scope="function")  # Default - runs for each test
def fresh_data():
    """New data for each test - ensures isolation."""
    return {"counter": 0}


@pytest.fixture(scope="module")
def expensive_resource():
    """Created once per test file - for expensive operations."""
    print("Creating expensive resource...")
    resource = {"data": "loaded from slow source"}
    yield resource
    print("Cleaning up expensive resource...")


@pytest.fixture(scope="session")
def database_connection():
    """Created once for entire test session."""
    print("Connecting to database...")
    connection = {"connected": True}  # Simulated connection
    yield connection
    print("Closing database connection...")

<br>

### Multiple conftest Files

---

You can have `conftest.py` at different directory levels:

```
tests/
├── conftest.py              # Available to ALL tests
│
├── unit/
│   ├── conftest.py          # Available only to unit tests
│   ├── test_models.py
│   └── test_utils.py
│
└── integration/
    ├── conftest.py          # Available only to integration tests
    ├── test_api.py
    └── test_database.py
```

In [ ]:
# tests/conftest.py - root level

import pytest

@pytest.fixture
def base_url() -> str:
    """API base URL - available to all tests."""
    return "http://localhost:8000"

In [ ]:
# tests/unit/conftest.py - unit tests only

import pytest

@pytest.fixture
def mock_database():
    """Mock database for unit tests - no real DB needed."""
    return {"users": [], "orders": []}

In [ ]:
# tests/integration/conftest.py - integration tests only

import pytest

@pytest.fixture(scope="module")
def test_database():
    """Real test database for integration tests."""
    # Setup test database
    db = create_test_database()
    yield db
    # Teardown
    db.drop_all()

<br>

## Test Functions vs Test Classes

---

### When to Use Functions

---

**Use standalone functions** for:
- Simple, independent tests
- Tests that don't share setup
- Quick one-off tests

In [ ]:
# tests/test_validators.py

def test_email_valid():
    assert validate_email("user@example.com") == True

def test_email_missing_at():
    assert validate_email("userexample.com") == False

def test_email_empty():
    assert validate_email("") == False

def test_username_valid():
    assert validate_username("john_doe") == True

def test_username_too_short():
    assert validate_username("ab") == False

<br>

### When to Use Classes

---

In [ ]:
# tests/test_script.py

import pytest
from script import validate_blog_post


class TestBlogPostAuthor:
    """Tests for author validation."""
    
    def test_valid_author(self):
        result = validate_blog_post("My Post", "John Doe", "Some clean content here.")
        assert result.is_valid == True
    
    def test_author_with_numbers(self):
        result = validate_blog_post("My Post", "John123", "Some clean content here.")
        assert result.is_valid == False


class TestBlogPostSpam:
    """Tests for spam detection."""
    
    @pytest.fixture
    def valid_base(self):
        return {"title": "My Post", "author": "John Doe"}
    
    def test_clean_content(self, valid_base):
        result = validate_blog_post(**valid_base, content="Great article about Django.")
        assert result.is_valid == True
    
    def test_spam_content(self, valid_base):
        result = validate_blog_post(**valid_base, content="buy now!")
        assert result.is_valid == False
        assert "Content looks like spam" in result.errors


class TestBlogPostMultipleErrors:
    """Tests for multiple validation errors."""
    
    def test_all_fields_invalid(self):
        result = validate_blog_post(title="", author="John123", content="buy now!")
        assert result.is_valid == False
        assert len(result.errors) == 3
    
    def test_errors_list_not_empty(self):
        result = validate_blog_post(title="", author="John Doe", content="Clean content.")
        assert result.errors != []

<br>

### Mixing Both

---

It's perfectly fine to mix functions and classes in the same file:

In [ ]:
# EXAMLPLE

# Simple standalone tests
def test_username_validation():
    assert is_valid_username("john_doe") == True


def test_email_validation():
    assert is_valid_email("john@example.com") == True


# Related tests grouped in class
class TestUserRegistration:
    """Tests for user registration flow."""
    
    def test_successful_registration(self):
        pass
    
    def test_duplicate_email_rejected(self):
        pass
    
    def test_weak_password_rejected(self):
        pass


class TestUserLogin:
    """Tests for user login."""
    
    def test_successful_login(self):
        pass
    
    def test_wrong_password(self):
        pass

<br>

## pytest Configuration

---

### pyproject.toml

---

The **recommended** way to configure pytest (modern Python projects):

In [ ]:
# Our pyproject.toml — the actual configuration we use:

# [tool.pytest.ini_options]
# testpaths = ["tests.py"]    ← tells pytest where to look for tests

**Our `pyproject.toml`:**

```toml
[project]
name = "kbc-django-copy"
version = "0.1.0"
requires-python = ">=3.14"

[dependency-groups]
dev = [
    "pytest>=9.0.3",
]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
```

`pythonpath = ["src"]` tells pytest to add `src/` to Python's import path — so `from script import ...` works without any path hacks.

The same options in `pyproject.toml` (modern equivalent):

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]
python_classes = ["Test*"]
python_functions = ["test_*"]
addopts = "-v --strict-markers"
markers = [
    "slow: marks tests as slow",
    "integration: integration tests",
]
```

`pytest.ini` is still supported, but `pyproject.toml` is the preferred approach for modern Python projects.

Alternative configuration file (legacy, but still supported):

```ini
# pytest.ini

[pytest]
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*
addopts = -v --strict-markers
markers =
    slow: marks tests as slow
    integration: integration tests
```

<br>

### Common Options

---

| Option | Description | Example |
|--------|-------------|----------|
| `testpaths` | Directories to search | `["tests"]` |
| `addopts` | Default command line options | `"-v --tb=short"` |
| `markers` | Custom markers definition | `["slow: ..."]` |
| `filterwarnings` | Warning filters | `["ignore::DeprecationWarning"]` |
| `python_files` | Test file patterns | `["test_*.py"]` |
| `norecursedirs` | Skip these directories | `[".venv", "node_modules"]` |

**Commonly used `addopts`:**

```toml
[tool.pytest.ini_options]
addopts = """
    -v
    --tb=short
    --strict-markers
    -ra
"""
```

| Flag | Effect |
|------|--------|
| `-v` | Verbose output |
| `--tb=short` | Shorter tracebacks |
| `--strict-markers` | Error on unknown markers |
| `-ra` | Show summary of all non-passing tests |

<br>

## Running Specific Tests

---

### By File and Directory

---

**Linux/macOS:**

```bash
# Run all tests
pytest

# Run tests in specific directory
pytest tests/unit/

# Run tests in specific file
pytest tests/test_users.py

# Run specific test function
pytest tests/test_users.py::test_create_user

# Run specific test class
pytest tests/test_users.py::TestUserRegistration

# Run specific method in class
pytest tests/test_users.py::TestUserRegistration::test_valid_email
```

**Windows:**

```powershell
# Run all tests
pytest

# Run tests in specific directory
pytest tests\unit\

# Run tests in specific file
pytest tests\test_users.py

# Run specific test function
pytest tests\test_users.py::test_create_user

# Run specific test class
pytest tests\test_users.py::TestUserRegistration

# Run specific method in class
pytest tests\test_users.py::TestUserRegistration::test_valid_email
```

<br>

### By Name Pattern

---

Use `-k` to filter by test name pattern:

```bash
# Run tests containing 'user' in name
pytest -k "user"

# Run tests containing 'user' AND 'valid'
pytest -k "user and valid"

# Run tests containing 'user' but NOT 'invalid'
pytest -k "user and not invalid"

# Run tests containing 'email' OR 'password'
pytest -k "email or password"
```

<br>

### By Marker

---

Mark tests with decorators, then filter:

In [ ]:
# tests/test_integration.py

import pytest


@pytest.mark.slow
def test_large_data_processing():
    """This test takes a long time."""
    pass


@pytest.mark.integration
def test_database_connection():
    """This test requires real database."""
    pass


@pytest.mark.integration
@pytest.mark.slow
def test_full_workflow():
    """Slow integration test."""
    pass


def test_fast_unit():
    """Fast unit test - no markers."""
    pass

**Running with markers:**

```bash
# Run only slow tests
pytest -m slow

# Run only integration tests
pytest -m integration

# Skip slow tests
pytest -m "not slow"

# Run integration but not slow
pytest -m "integration and not slow"
```

**Built-in markers:**

| Marker | Purpose |
|--------|---------|
| `@pytest.mark.skip` | Skip test unconditionally |
| `@pytest.mark.skipif(condition)` | Skip if condition is true |
| `@pytest.mark.xfail` | Expected to fail |
| `@pytest.mark.parametrize` | Run with multiple inputs |

In [ ]:
import pytest
import sys


@pytest.mark.skip(reason="Not implemented yet")
def test_future_feature():
    pass


@pytest.mark.skipif(sys.platform == "win32", reason="Unix only")
def test_unix_specific():
    pass


@pytest.mark.xfail(reason="Known bug #123")
def test_known_issue():
    assert False  # This failure is expected

## What's Next: Testing a Django Project

---

So far we've tested pure Python functions — no framework, no database, no HTTP.

That's intentional: the fundamentals (assertions, fixtures, structure) are the same regardless of what you're testing.

Now we'll apply exactly these skills to a real Django project. The differences are mostly about **setup**:

- Django needs to be initialized before tests can run (`django.setup()`)
- Models talk to a database — tests need a test database
- Views handle HTTP requests — tests use a test client

The pytest knowledge you have now transfers directly. The next lessons cover the Django-specific wiring.

| Concept | Key Points |
|---------|------------|
| **Directory structure** | `tests/` directory, mirror source structure |
| **conftest.py** | Shared fixtures, auto-discovered, multi-level |
| **Fixture scope** | `function`, `class`, `module`, `session` |
| **Functions vs classes** | Functions for simple tests, classes for grouping |
| **Configuration** | `pyproject.toml` (recommended) or `pytest.ini` |
| **Running tests** | `-k` for name pattern, `-m` for markers |

<br>

**Key principles:**

1. **Mirror your source structure** in tests
2. **Use conftest.py** for shared fixtures
3. **Choose appropriate fixture scope** for performance
4. **Group related tests** in classes
5. **Use markers** to categorize tests
6. **Configure pytest** in `pyproject.toml`

You're given a flat test file. Reorganize it into a proper structure.

<br>

**Original messy test file** (`test_everything.py`):

<br>

**Your task:**

1. Create a proper directory structure with:
   - `tests/` directory
   - `tests/unit/` for unit tests
   - `tests/integration/` for integration tests
   - Appropriate `conftest.py` files

2. Split tests into separate files:
   - `test_product.py` - product-related tests
   - `test_user.py` - user-related tests
   - `test_order.py` - order-related tests

3. Create shared fixtures in `conftest.py`

4. Add a `pyproject.toml` with pytest configuration

<details>
    <summary>▶️ Solution</summary>

**Directory structure:**

```
my_project/
├── src/
│   └── script.py
├── tests/
│   ├── conftest.py
│   ├── test_blog.py
│   └── test_validators.py
└── pyproject.toml
```

---

**tests/conftest.py:**

```python
import pytest


@pytest.fixture
def valid_post():
    return {
        "title": "Django Testing Guide",
        "author": "John Doe",
        "content": "This is a great blog post about Django testing.",
    }
```

---

**tests/test_blog.py:**

```python
import pytest
from script import validate_blog_post


class TestBlogAuthor:
    def test_valid_author(self, valid_post):
        result = validate_blog_post(**valid_post)
        assert result.is_valid == True

    def test_author_with_numbers(self, valid_post):
        valid_post["author"] = "John123"
        result = validate_blog_post(**valid_post)
        assert result.is_valid == False
        assert "Author must contain only letters and spaces" in result.errors


class TestBlogSpam:
    def test_spam_content(self, valid_post):
        valid_post["content"] = "buy now!"
        result = validate_blog_post(**valid_post)
        assert result.is_valid == False

    def test_multiple_errors(self):
        result = validate_blog_post(title="", author="John123", content="buy now!")
        assert len(result.errors) == 3
```

---

**tests/test_validators.py:**

```python
import pytest
from script import validate_author_name


def test_valid_author_name():
    validate_author_name("John Doe")  # no exception


def test_author_with_numbers_raises():
    with pytest.raises(ValueError, match="Only letters"):
        validate_author_name("John123")
```

---

**pyproject.toml:**

```toml
[project]
name = "my-project"
version = "0.1.0"

[dependency-groups]
dev = ["pytest>=9.0.0"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
addopts = "-v --strict-markers"
```

</details>

---